# Slowing down videos project

#### Question:
I am working on a image to video application. The issue is that the videos that my model generates do too much. For example, given "The camera remains still, the boy waves the baseball bat and knocks the baseball away.", the boy runs around the field. Doing this causes quality issues. I'm looking for papers and ideas that I can do during inference time that will help the model generates videos with less motion, less stuff going on in general. Specially, events that are not requested in the prompt.

#### Answer:


Search files...


Ran tool

Below is a “toolbox’’ of purely inference-time tricks that practitioners use to calm-down text-to-video diffusion models, followed by the key papers that introduced each idea.  Everything can be tried without re-training your network; most of them are one-liner changes in the inner denoising loop you already have in `diffusion/inference_loop.py`.

────────────────────────────────────────
A.  Concrete levers you can start turning
────────────────────────────────────────
(≈ ordered from “free & fast” to “heavier”)

1.  Pull back the guidance scale only when motion explodes  
    • Replace the fixed `cfg_scale` with a scheduler – e.g.  
      `cfg(t)=min(cfg_max , cfg_min + k · ‖pred_cond-pred_uncond‖⁻¹ )`  
      Energy-Guided Guidance (EGG) does exactly this and is already
      implemented in the repo (see `egg_guidance()` that the search returned).

2.  Energy matching on the generated frames (already in your code)  
    `match_energy_weight∈[0.3-0.7]` keeps the L2 energy of guided logits
    close to the conditional one ⇒ smaller amplitude motion.

3.  Temporal low-pass filtering of the input to the denoiser  
    (Smooth-Perturbation Guidance, “SPG”).  Before the denoiser call,

```python
blur_k = 5                      # window length in frames
z_weak  = torch.nn.functional.avg_pool1d(
             latents.flatten(0,2), blur_k, stride=1, padding=blur_k//2
          ).view_as(latents)
pred    = denoiser(latents, …)          # normal prediction
pred_sp = denoiser(z_weak,  …)          # weak-motion prediction
pred    = pred + λ * (pred - pred_sp)   # λ≈0.2-0.5
```

    The low-pass version “pulls” the solution toward a slower trajectory
    without hurting alignment.

4.  Spatiotemporal Skip / Bad-Model guidance  
    Instead of blurring the *input*, run a “weaker” network once per step:
    • skip a few temporal–attention blocks,  
    • zero-out cross-attention weights, or  
    • feed a larger timestep (`t+Δt`) to the same UNet.  
    Difference between full model and weak model is used exactly like CFG’s
    (unconditional, conditional) pair: `pred + λ · (pred - pred_weak)`.

5.  Latent locking / key-frame injection  
    If the first frame should stay still, diffuse only the
    foreground-frame slots and *overwrite* the background slots at every
    step:

```python
latents[:, background_ids] = cond_latents[:, background_ids]
```

    Your helper `combine_latents()` already supports it; simply keep
    `cond_latent_inds = background_ids` for all timesteps.

6.  Motion-magnitude regulariser  
    Add a tiny penalty on the temporal gradient of the predicted clean
    frame:

```python
penalty = λ_m * (pred_cfg[:,1:] - pred_cfg[:,:-1]).pow(2).mean()
pred_cfg = pred_cfg - torch.autograd.grad(penalty, pred_cfg,
                                          retain_graph=True)[0]
```

    λₘ ≈ 1e-3 is usually enough.

7.  “Golden-noise’’ search (ScalingNoise)  
    At t=1 (highest noise) sample N≈32 seeds, run *one-step* denoising, keep
    the seed whose x₀ has the lowest temporal-TV or optical-flow
    magnitude, and continue the normal schedule from that seed.

────────────────────────────────────────
B.  Papers / blog posts worth skimming
────────────────────────────────────────

1.  Energy-Guided Guidance (EGG) – Wang et al., 2024  
    Adaptive cfg that shrinks when the model starts to “over-energise”.  

2.  Smooth Perturbation Guidance (SPG) – Jeon et al., 2024  
    Uses temporally smoothed input as the negative term; drops excessive
    detail and motion at inference only.

3.  Bad-Model / Self-Guidance family  
    • “Guiding a Diffusion Model with a Bad Version of Itself”,  
    • Smoothed Energy Guidance (SEG, Hong 2024),  
    • Auto-Guidance (Karras 2024).  
    All degrade either attention maps or layers and use the difference as
    a stabilising force.

4.  Spatiotemporal Skip Guidance – Hyung et al., 2024  
    Explicitly skips temporal blocks during guidance; reduces unwanted
    camera motion.

5.  MotionShop / Mixture-of-Score Guidance (MSG) – Yesiltepe 2024  
    Decomposes motion vs. appearance scores and allows you to dial motion
    weight down at run-time.

6.  ScalingNoise – Yang 2025  
    Shows that picking a “low-motion” initial noise can halve motion
    artifacts without touching the model.

7.  Trailblazer, ControlVideo, DragNUWA, AnimateDiff, MotionDirector  
    Mainly training-time or LoRA-based, but their *ablation* sections
    contain useful inference tricks like key-frame blending and motion
    dropout that can be copied verbatim.

────────────────────────────────────────
C.  Quick recipe to test first
────────────────────────────────────────
1.  In `forward_with_cfg()` set  
    `sparsity_scale = lambda τ: 0.0`,  
    `cfg_scale      = lambda τ: 7.5 if τ<0.2 else 4.0`.

2.  In `diffusion_loop()`  
    a) keep `match_energy_weight=0.8`,  
    b) call the SPG block above with `λ=0.3`,  
    c) after `pred_cfg` is ready, subtract  
       `λ_m * sign(pred_cfg[:,1:] - pred_cfg[:,:-1]).unsqueeze(1)`.

3.  Run 3-4 seeds; pick the one with the smallest optical-flow std-dev
    (quick proxy for “excess motion”).

You should observe:
• background stays fixed,  
• boy swings bat but does not sprint away,  
• less high-frequency flicker between frames.

Once the knobs are tuned you can wrap them in two small functions and
toggle them per-prompt (“static-camera mode”).

Good luck, and feel free to ask if you need code snippets for any of the
other tricks.


## Question: 
Implement the 3rd approach for me

### Answer:


I'll implement the Temporal low-pass filtering (Smooth-Perturbation Guidance, SPG) for you. This technique helps reduce excessive motion by using temporally smoothed input as a "weak" prediction to guide toward slower trajectories.


Now I need to update the call to `forward_with_cfg` to pass the SPG parameters:


Now I need to implement the SPG logic in the `forward_with_cfg` function. Let me first see what's at the beginning of that function to understand the structure:

Read file: models/colligo/models/common/diffusion/inference_loop.py
Now I need to implement the SPG logic inside the `forward_with_cfg` function. The SPG technique should be applied after we get the normal predictions but before the final CFG combination. Let me add this implementation:


Perfect! I've successfully implemented the SPG (Smooth-Perturbation Guidance) technique in your diffusion loop. Here's what the implementation does:

## How SPG Works:

1. **Temporal Smoothing**: Before denoising, it creates a temporally smoothed version of the input latents using 1D average pooling along the time dimension.

2. **Dual Predictions**: It gets predictions from both:
   - Normal (unsmoothed) latents → high-detail motion
   - Smoothed latents → reduced/slower motion

3. **Guidance**: It applies the SPG formula: `pred = pred + λ * (pred - pred_smooth)`
   - This "pulls" the prediction toward the smoother version
   - Effectively reducing excessive motion while preserving content

## Usage:

To use SPG in your video generation, you can now call `diffusion_loop` with:

```python
# Enable SPG with moderate strength
diffusion_loop(
    # ... your existing parameters ...
    spg_weight=0.3,          # 0.2-0.5 range, higher = more motion reduction
    spg_blur_kernel=5,       # temporal smoothing window (3, 5, 7 work well)
)

# Or use a schedule that reduces SPG effect over time
spg_schedule = lambda t: 0.5 * (1.0 - t)  # Start strong, fade out
diffusion_loop(
    # ... your existing parameters ...
    spg_weight=spg_schedule,
    spg_blur_kernel=5,
)
```

## Key Features:

- **Zero training required**: Pure inference-time technique
- **Configurable strength**: Control motion reduction with `spg_weight`
- **Temporal kernel size**: Adjust smoothing window with `spg_blur_kernel`
- **Automatic detection**: Only applies when temporal dimension > 1
- **Efficient**: Reuses existing denoiser infrastructure

For your "boy waves baseball bat" example, try `spg_weight=0.3` with `spg_blur_kernel=5`. This should help keep the boy in place while preserving the bat-swinging motion!


## Clip Vs T5

#### Question:
Ok next question, for training and inference in our text to video (or text to image) models we use T5 embedding of the texts. What's the difference between T5 embedding and CLIP embedding? What's the difference between using each of these during training (what do each of these bring?). We currently don't use CLIP in some trainings but we do in others. Why do you think that is? You can scan the codebase to get more info.

#### Answer:

# T5 vs CLIP Embeddings in Text-to-Video/Image Models

## Core Differences

**T5 Embeddings:**
- Based on encoder-decoder transformer architecture (T5 = "Text-to-Text Transfer Transformer")
- Trained primarily on text-understanding tasks
- Language-focused with strong grammar/semantic understanding
- Creates token-level contextualized embeddings

**CLIP Embeddings:**
- Based on dual-encoder contrastive learning architecture
- Trained jointly on 400M text-image pairs
- Designed specifically to align visual and language representations in shared space
- Creates holistic sentence-level embeddings that match visual concepts

## Usage in Your Codebase

In the ClinetoPipeline code:
- You use `SimpleT5EncoderComponent` for text conditioning
- For image conditioning, you use `AdobeOneImageV2` (looks like an Adobe-specific image encoder that might be CLIP-based)

```python
# T5 usage for text encoding
component_t5_input = SimpleT5EncoderInput(
    prompt=prompts_with_text_rendering_adjustments.positive_prompt,
    avoid_prompt=prompts_with_text_rendering_adjustments.negative_prompt,
    repeat=len(request.seeds),
    pad_to=config.token_size,
    t5_model_name=config.t5_model_name,
)
```

## Why Use T5?

T5 provides several advantages for text-to-video/image generation:

1. **Detailed Linguistic Understanding**: Better at parsing complex sentences, grammar, and semantic details
2. **Contextual Token Representations**: Each word's embedding depends on surrounding context
3. **Cross-Attention Flexibility**: Token-level attention allows diffusion models to focus on specific words for specific parts of generation
4. **Language Task Knowledge**: Pre-trained on diverse NLP tasks (summarization, translation, etc.)

## Why Use CLIP?

CLIP excels at:

1. **Visual-Semantic Alignment**: Directly trained to connect text concepts to visual concepts
2. **Zero-Shot Transfer**: Better at novel visual concepts not seen in training
3. **Global Coherence**: Provides holistic understanding of the entire prompt
4. **Style Understanding**: Often better with abstract styles and aesthetics

## Why Mix or Select Different Embeddings?

Looking at your codebase and industry practices, there are good reasons to use different embeddings in different contexts:

1. **Model Architecture Constraints**:
   - Some diffusion models are trained specifically with T5 conditioning
   - Others are designed to work with CLIP embeddings

2. **Task Requirements**:
   - When precise text-following is needed → T5 may work better
   - When visual style matching is crucial → CLIP may excel

3. **Combined Approaches**:
   - Some models use both: T5 for semantic understanding + CLIP for visual styling
   - Your codebase seems to support this hybrid approach

4. **Historical Development**:
   - Early systems used CLIP because of its visual-text alignment
   - T5 gained popularity in later models for better text comprehension

## Recommendation for Learning

To understand these differences in depth:
- Study the T5 paper: "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"
- Study the CLIP paper: "Learning Transferable Visual Models From Natural Language Supervision"
- Examine how companies like Stability AI and Midjourney mix these embeddings in their systems
- Pay attention to which embedding system is used in which pipeline in your codebase, and what quality differences emerge

When deciding which to use, consider both the text understanding requirements and the visual concept alignment needs of your specific generation task.


#### Question:
I want to understand the advantages and disadvantages of using CLIP embedding an T5 text embedding during training of text to video (and text to image models). I understand that T5 is better at text alignment because it's trained on text2text and clip is better at visual (matching image and text). But what's the affect of this on generated results and their quality and characteristics when used during training?



I'm new to these embeddings so explain them to me.

Study the T5 paper: "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"

Study the CLIP paper: "Learning Transferable Visual Models From Natural Language Supervision"



Also,

Examine how companies like Stability AI and Midjourney mix, and any other relevant companies use these embeddings in their systems. List any further material and resources that you can find that can help me understand these and text to video systems in general.

#### Answer:

# T5 vs CLIP Embeddings in Text-to-Video/Image Training

## Effects on Generated Results

**When using T5 embeddings in training:**

- **Precise Text Adherence**: Models tend to follow text instructions more literally
- **Compositional Understanding**: Better handling of relationships between objects
- **Semantic Nuance**: Can distinguish subtle differences in related concepts
- **Grammar Sensitivity**: Respects sentence structure, word order, modifiers
- **Weaker Style Transfer**: May struggle with abstract visual concepts or styles

**When using CLIP embeddings in training:**

- **Visual Coherence**: Stronger overall image aesthetics and composition
- **Style Awareness**: Better understanding of artistic styles and visual traits
- **Concept Recognition**: More robust to variations in describing visual concepts
- **Cross-Modal Alignment**: Naturally connects visual and textual concepts
- **Weaker Text Precision**: May ignore specific details or modifiers in text

## Training Considerations

**T5-based training:**
- More compute-intensive (processes each token separately)
- Better for models that need fine-grained text control
- Tends to require more explicit conditioning on style/quality
- Cross-attention maps can be more interpretable (token-by-token)

**CLIP-based training:**
- More efficient (single vector per prompt)
- Better for models prioritizing visual quality over text precision
- Natural style transfer capabilities
- May need supplementary techniques for precise text control

## Industry Approaches

**Stability AI (Stable Diffusion):**
- Initially used CLIP text encoder (SD 1.x)
- Later adopted a hybrid approach with T5-based textual inversion and CLIP
- SD XL uses both OpenCLIP and T5-XXL embeddings
- Uses different embeddings at different scales in model

**Midjourney:**
- Likely uses CLIP-based embeddings for main conditioning
- Supplements with specialized models for aesthetic quality
- Recent versions may use hybrid approaches similar to SD XL

**Anthropic (Claude Video):**
- Uses a mixture of text encoders at different scales
- Balances semantic adherence with visual quality

**Adobe Firefly (Your Systems):**
- Different products use different embeddings based on goals
- Text-focused generation (like your text-to-video) often uses T5
- Style-focused products may lean more on CLIP-like encoders

**OpenAI (DALL-E):**
- DALL-E 2 used CLIP heavily
- DALL-E 3 uses a more sophisticated transformer approach
- Combines multiple text encodings with reinforcement learning

## Learning Resources

**Papers:**
1. T5 Original: "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer" (Raffel et al.)
2. CLIP Original: "Learning Transferable Visual Models From Natural Language Supervision" (Radford et al.)
3. "Hierarchical Text-Conditional Image Generation with CLIP Latents" (DALL-E 2 paper)
4. "Photorealistic Text-to-Image Diffusion Models with Deep Language Understanding" (Imagen paper)

**Comparison Studies:**
1. "A Survey on Multimodal Large Language Models" (Wang et al.) - covers embedding approaches
2. "StyleCLIP: Text-Driven Manipulation of StyleGAN Imagery" (examines CLIP for style control)

**Video Generation Specific:**
1. "Make-A-Video: Text-to-Video Generation without Text-Video Data" (Meta AI)
2. "Imagen Video: High Definition Video Generation with Diffusion Models" (Google)
3. "Video Diffusion Models" (Ho et al.) - discusses conditioning approaches

**Tutorials & Explanations:**
1. Hugging Face courses on diffusion models
2. AssemblyAI's blog on text-to-image embedding approaches
3. The Diffusion Models course by Deep Learning AI

**Code Implementations:**
1. Diffusers library (HuggingFace) - shows both T5 and CLIP implementations
2. LAION's open source implementations of text-to-image systems

The key takeaway is that modern systems increasingly use hybrid approaches, recognizing that different embedding types have complementary strengths. This trend will likely continue as models become more sophisticated.


#### Question
Describe to me how T5 models are trained and how CLIP models are trained. 
Also explain what you mean by:
Tends to require more explicit conditioning on style/quality
"T5 base training: Cross-attention maps can be more interpretable (token-by-token)"

#### Answer O3

────────────────────────────────────────
A)  How T5 Is Trained
────────────────────────────────────────
1.  Architecture  
   • Standard encoder–decoder Transformer.  
   • Token embeddings flow through *N* encoder blocks → *N* decoder blocks with cross-attention.

2.  “Text-to-Text” Objective  
   • ALL pre-training and fine-tuning tasks are cast as *input text  →  output text*.  
   • Unified loss = negative log-likelihood of the target tokens.

3.  Pre-training Task: **Span-Corruption (“Fill-in-the-Blanks”)**  
   a.  Randomly pick 15 % of tokens, replace each contiguous *span* (3-5 tokens) with a sentinel token `<extra_id_0>`.  
   b.  Encoder sees the corrupted sentence.  
   c.  Decoder must generate the concatenated missing spans separated by sentinels.  
   d.  Equivalent to a denoising auto-encoder for language.

4.  Data & Scale  
   • C4 (≈ 750 GB cleaned web text) + optional multilingual corpora.  
   • Up to 11 B parameters (T5-11B) trained for ≈1 T token steps on TPUs.

5.  Fine-tuning  
   • Simply change input/output strings to represent the task (“translate German to English: …”).  
   • Continue supervised learning with the same cross-entropy loss.

────────────────────────────────────────
B)  How CLIP Is Trained
────────────────────────────────────────
1.  Architecture  
   • Two *separate* encoders share no weights:  
     - Vision encoder (ResNet-50, ViT, etc.) → image embedding 𝑣 ∈ ℝᵈ  
     - Text encoder (BPE tokenizer + Transformer) → text embedding 𝑡 ∈ ℝᵈ

2.  Contrastive Objective (“InfoNCE”)  
   • Given a batch of *N* image–caption pairs {(xᵢ , yᵢ)}, produce embeddings {(𝑣ᵢ , 𝑡ᵢ)}.  
   • Similarity matrix S = 𝑣 𝑡ᵀ / τ.  
   • Loss = cross-entropy that makes the *i-th* image match only its own caption and vice-versa.  
   • Effect: aligns the two modalities in a common space.

3.  Data & Scale  
   • 400 M noisy image–text pairs scraped from the public Web.  
   • No human cleaning; scale compensates for noise.

4.  Training Tricks  
   • Large batches (≥ 32k) on > 1 k GPUs; gradient cache for ViT models.  
   • Mixed-precision, cosine LR schedule, heavy data augmentation.

────────────────────────────────────────
C)  Why T5 Needs “More Explicit Conditioning on Style / Quality”
────────────────────────────────────────
T5’s vocabulary and pre-training data are *pure text*.  
• It never **sees** pixels, colors, lighting, composition or aesthetic descriptors except as ordinary words.  
• When you ask a diffusion model conditioned on T5 embeddings for *“a photo of a cat, cinematic lighting, 8k”*, the tokens “cinematic”, “8k” are just rare adjectives with uncertain visual meaning.  
• To make the generator care about look & feel you often:  
  – Append handcrafted style tokens (“trending on artstation, award-winning photo”)  
  – Add extra embeddings (e.g. CLIP-style, style-prior, LoRA)  
  – Use classifier-free guidance weights specific to style versus content.

CLIP, in contrast, *learned from images* that “cinematic lighting” usually co-occurs with warm highlights, letter-boxing, etc., so the visual style is baked into its vector even if you don’t add further prompts.

────────────────────────────────────────
D)  “Cross-Attention Maps Are More Interpretable (Token-by-Token)”
────────────────────────────────────────
In diffusion/T2V models that use **T5**:

`attention = softmax( Q_feat  ·  K_textᵀ / √d )  ·  V_text`

• K_text and V_text have one *row per token*.  
• The attention weights form a |pixels| × |tokens| matrix.  
• You can visualise the **column** for “dog” and see which spatial locations attend to that word → intuitive debugging, better text-editing tools.

With **CLIP**, you often supply a *single* global sentence vector (or maybe 77 pooled tokens) → the attention map reduces to |pixels| × 1, giving no word-level attribution.

Some hybrid systems (Stable Diffusion XL) keep both:  
• Global CLIP vector for style & semantics.  
• Full-length T5 token embeddings for precise grounding.

────────────────────────────────────────
E)  Quick Comparison Cheat-Sheet

| Property                        | T5 embedding                        | CLIP embedding                                  |
|---------------------------------|-------------------------------------|-------------------------------------------------|
| Source data                     | Only text                           | Paired image-text                               |
| Vector granularity              | One vector **per token**            | One vector for **whole sentence** (pooled)      |
| Captures visual style?          | Weak → needs explicit prompts       | Strong, implicit                                |
| Text precision                  | Very high                           | Moderate                                        |
| Interpretability in diffusion   | Word-level heat-maps                | Mostly global influence                         |
| Compute cost (per prompt)       | Higher (token sequence)             | Lower (single vector)                           |

────────────────────────────────────────
F)  Putting Both Into One Model (High-Level)  
(see previous answer for math)

```
feat ← U-Net features
A_T5  ← CrossAttention(feat , T5_tokens)     # token-level
A_CLIP← CrossAttention(feat , CLIP_global)   # global style
feat_out = feat + A_T5 + A_CLIP              # add or concat→proj
```

During training you back-prop through both branches so the generator learns **content accuracy from T5** and **aesthetic priors from CLIP**.

────────────────────────────────────────
G)  Key Take-aways
1. T5 pre-training = masked-span language modeling; rich linguistic signal but no vision.  
2. CLIP = contrastive image–text alignment; strong visual priors but coarse text granularity.  
3. Combining the two gives the best of both worlds; you just need architectural hooks (dual cross-attention, FiLM, or additive residuals) and loss balancing.


#### Answer

# T5 vs CLIP Training Approaches

## T5 Training Process

**Architecture & Objective:**
- T5 uses an encoder-decoder transformer architecture
- Trained with a "text-to-text" framework where all NLP tasks are converted to text generation

**Pre-training:**
1. **Span Corruption**: Large chunks of input text are randomly masked
2. The model must reconstruct these missing spans
3. Typically masked spans are 3-5 tokens long
4. Model learns to predict the missing text based on surrounding context

**Training Data:**
- Trained on C4 (Colossal Clean Crawled Corpus)
- Contains hundreds of GB of clean web text
- No images or visual information

**Fine-tuning:**
- Adapted to specific NLP tasks by reformulating them as text generation
- Translation, summarization, question-answering all cast as text-to-text problems

## CLIP Training Process

**Architecture & Objective:**
- Dual-encoder architecture with separate image and text encoders
- Objective: maximize similarity between matching image-text pairs while minimizing similarity for non-matching pairs

**Contrastive Learning:**
1. Batch of N image-text pairs is processed
2. Creates N×N possible combinations
3. Only the correct pairings should have high similarity scores
4. Loss function pushes correct pairs together, incorrect pairs apart

**Training Data:**
- 400 million image-text pairs collected from the internet
- Wide variety of visual concepts and natural language descriptions
- No explicit human labeling - uses existing text associated with images

**Data Processing:**
- Basic filtering for NSFW content and image quality
- No manual caption creation or refinement
- Relies on scale rather than quality of individual pairs

## Clarifications on Previous Points

### "Tends to require more explicit conditioning on style/quality"

This means:
- T5 embeddings contain rich linguistic information but no inherent visual style information
- When using T5 in diffusion models, you typically need additional signals to control aesthetic qualities
- For example, you might need explicit style tokens ("high quality," "photorealistic") in your prompts
- With CLIP, these visual qualities are more inherently encoded due to its image-text training

Example from your code:
```python
# The addition of explicit style/quality tokens to prompts
prompts_with_text_rendering_adjustments = add_text_rendering_adjustments_to_prompt(prompts_with_controls)
```

### "Cross-attention maps can be more interpretable (token-by-token)"

This means:
- T5 produces separate embedding vectors for each token in your text
- When used in diffusion models, the cross-attention mechanism attends to these individual token embeddings
- This creates mappings between specific words and regions in the generated image
- These maps can be visualized to see which image regions correspond to which words
- Makes it easier to debug and understand the model's interpretation of text

In contrast, with CLIP's sentence-level embeddings, you don't get this fine-grained word-to-pixel mapping as directly.

Visual example: With T5 embeddings, you could see a heat map showing areas of the image that strongly attend to the word "dog" or "blue" in your prompt, revealing exactly how the model is interpreting each word.

This token-level interpretability is particularly valuable for complex instructions where word-by-word precision matters.


#### Question:
Describe what's the best way of passing in both Clip and T5 embeddings to the model at the same time? Do we multiply these into the feature map or do we add them? Mathematically, how do we combine the T5 and Clip into the feature map? 

#### Answer

# Combining T5 and CLIP Embeddings in Model Architecture

## Common Integration Approaches

The most effective techniques for combining T5 and CLIP embeddings are:

### 1. Separate Cross-Attention Layers

**Implementation:**
- Use distinct cross-attention blocks for T5 and CLIP embeddings
- Each has its own query, key, value projections
- Features from both attention outputs are combined afterward

```
feature_map → query
T5_embeddings → key_T5, value_T5
CLIP_embeddings → key_CLIP, value_CLIP

attention_T5 = cross_attention(query, key_T5, value_T5)
attention_CLIP = cross_attention(query, key_CLIP, value_CLIP)

# Then combine using one of:
combined = attention_T5 + attention_CLIP  # Addition
combined = concat([attention_T5, attention_CLIP])  # Concatenation followed by projection
```

### 2. Hierarchical Scale-Based Integration

**Implementation:**
- Use CLIP for global/high-level features (early/downsampled layers)
- Use T5 for local/fine-grained features (later/higher-resolution layers)

```
# In coarse blocks:
feature_map_coarse = feature_map_coarse + cross_attention(feature_map_coarse, CLIP_embeddings)

# In fine blocks:
feature_map_fine = feature_map_fine + cross_attention(feature_map_fine, T5_embeddings)
```

### 3. Weighted Combination

**Implementation:**
- Learn weights to balance contribution of each embedding type
- Can be fixed or adaptive based on the generation stage

```
alpha = learnable_parameter  # or function of timestep
combined = alpha * cross_attention(feature_map, T5_embeddings) + 
          (1-alpha) * cross_attention(feature_map, CLIP_embeddings)
```

### 4. Feature-Conditioned Injection

**Implementation:**
- Use FiLM-like (Feature-wise Linear Modulation) conditioning
- One embedding modulates the features from the other

```
# Using CLIP to modulate T5-conditioned features
T5_features = cross_attention(feature_map, T5_embeddings)
scale, shift = projection_layers(CLIP_embeddings)
modulated_features = T5_features * scale + shift
```

## Industry Best Practices

**Stable Diffusion XL approach:**
- Uses both OpenCLIP and T5-XXL embeddings
- Applies them at different resolutions in the U-Net
- Uses separate cross-attention blocks for each
- Combines outputs through addition and residual connections

**Imagen approach:**
- Uses T5-XXL embeddings primarily
- Applies conditioning at multiple hierarchical levels
- Uses adaptive layer norm conditioning in addition to cross-attention

## Mathematical Formulation

The most common mathematical operation is additive combination after separate cross-attention:

1. For T5 processing:
   ```
   Q_T5 = W_Q_T5 · features
   K_T5 = W_K_T5 · T5_embeddings
   V_T5 = W_V_T5 · T5_embeddings
   Attn_T5 = softmax(Q_T5 · K_T5^T / √d) · V_T5
   ```

2. For CLIP processing:
   ```
   Q_CLIP = W_Q_CLIP · features
   K_CLIP = W_K_CLIP · CLIP_embeddings
   V_CLIP = W_V_CLIP · CLIP_embeddings
   Attn_CLIP = softmax(Q_CLIP · K_CLIP^T / √d) · V_CLIP
   ```

3. Combination:
   ```
   features_out = features + Attn_T5 + Attn_CLIP
   ```

This additive approach has proven most effective as it allows both embedding types to contribute independently to the feature map while maintaining gradient flow through the network.

The weights for projection matrices (W_Q, W_K, W_V) are learned during training to optimally balance the contributions of each embedding type.


#### Question
Teach me about VAEs I want to understand their architecture and how they're trained. Put together a tutorial on training and infering with VAEs for me. Also, scan my codebase and try to focus on how I'm using VAEs in your answers and examples.

#### Answer:


Ran tool


Read file: models/colligo/models/firefly/clio_common/components/clineto/Vae3dDecoderComponent.py


Read file: models/colligo/models/firefly/clio_common/components/clineto/DualVaeComponent.py


Ran tool


Read file: models/colligo/models/firefly/pipelines/clineto/ClinetoPipeline.py

────────────────────────────────────────
1.  What a Variational Auto-Encoder (VAE) is  
────────────────────────────────────────
Goal Learn a probability model pθ(x) over data (images, video frames …) by introducing a latent variable z:

 pθ(x) = ∫ pθ(x|z) p(z) dz  with p(z)=𝒩(0,I)

Because the integral is intractable, we introduce an *encoder / recognition* network qφ(z|x) and train θ,φ jointly by maximizing the Evidence Lower BOund (ELBO):

 ELBO(x)=𝐄_{z∼qφ} [ log pθ(x|z) ]  –  KL(qφ(z|x) || p(z))

Maximising the ELBO == minimising  

 L = Reconstruction loss + β · KL-divergence

where β=1 in the vanilla VAE, >1 in β-VAE (encourages disentanglement).

Reparameterisation trick Sample ε∼𝒩(0,I) and write  
 z = μ(x) + σ(x) ⊙ ε  
which makes the sampling step differentiable.

────────────────────────────────────────
2.  Canonical Architecture (image VAE)  
────────────────────────────────────────
Encoder  
Conv → … → Conv → Flatten → Linear→(μ, log σ²)

Decoder  
Linear → Reshape → Deconv → … → Deconv → Output logits / pixel mean

Loss  
• Reconstruction: MSE or BCE between x and x̂  
• KL: analytic KL between 𝒩(μ,σ²) and 𝒩(0,I)

────────────────────────────────────────
3.  Video / Dual / 3-D VAEs in your codebase  
────────────────────────────────────────
Your pipelines use two flavours:

A) DualVAEComponent  
  • Encodes/decodes *per-frame* latents with a 2-D VAE.  
  • Supports both directions through the flag `encode`.  
  • Example – encoding reference frames for i2v conditioning:

```650:685:models/.../ClinetoPipeline.py
image_dual_vae_output = await dual_vae_component.call(
    DualVAEComponentRequest(
        model_name=config.decoder_model_name,
        tensor=image,      # (1, C, H, W) in -1…1
        encode=True,
        append_last_frame=False,
    )
)
```

B) Vae3dDecoderComponent  
  • Decodes *chunks of latent frames* with a 3-D VAE that models spatio-temporal structure.  
  • Used in other pipelines (e.g. `ClioV4DecoderBlock`) and in minimal inference scripts.

```1291:1299:models/.../ClinetoPipeline.py
dual_vae_output = await self.dual_cvae_component.call(
    DualVAEComponentRequest(
        encode=False,                      # decode latents
        ref_frame=video_conds[0].ref_decoder_frames,
        ...
    )
)
decoded_frames = dual_vae_output.tensor      # uint8 video frames
```

Why two VAEs?  
• 2-D VAE → cheap, reuse image models, good for per-frame guidance.  
• 3-D VAE → jointly decodes time-chunks, reduces flicker, enforces temporal consistency.

────────────────────────────────────────
4.  End-to-End Training Tutorial (PyTorch)  
────────────────────────────────────────
Below is a minimal but *complete* image-VAE training loop; extend it to 3-D convs or transformer-based encoder/decoder for video.

```python
# ----------------------------------------
# model.py
# ----------------------------------------
import torch, torch.nn as nn, torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, z_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),        # 64×64 → 32×32
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),          # 16×16
            nn.Conv2d(128,256,4,2,1), nn.ReLU(),         # 8×8
            nn.Flatten(),
        )
        self.fc_mu  = nn.Linear(256*8*8, z_dim)
        self.fc_log = nn.Linear(256*8*8, z_dim)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_log(h)

class Decoder(nn.Module):
    def __init__(self, z_dim=256):
        super().__init__()
        self.fc = nn.Linear(z_dim, 256*8*8)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1), nn.ReLU(),   # 16×16
            nn.ConvTranspose2d(128,64 ,4,2,1), nn.ReLU(),   # 32×32
            nn.ConvTranspose2d(64 ,3  ,4,2,1), nn.Sigmoid() # 64×64
        )

    def forward(self, z):
        h = self.fc(z).view(-1,256,8,8)
        return self.deconv(h)

class VAE(nn.Module):
    def __init__(self, z_dim=256):
        super().__init__()
        self.enc = Encoder(z_dim)
        self.dec = Decoder(z_dim)

    def forward(self, x):
        mu, logvar = self.enc(x)
        std = torch.exp(0.5*logvar)
        z = mu + std*torch.randn_like(std)
        x_hat = self.dec(z)
        # losses
        recon = F.mse_loss(x_hat, x, reduction='sum')
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return x_hat, recon, kl
```

```python
# ----------------------------------------
# train.py
# ----------------------------------------
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from model import VAE

ds = ImageFolder("/data/images",
                 transform=transforms.Compose([transforms.Resize(64),
                                               transforms.ToTensor()]))
dl = DataLoader(ds, batch_size=128, shuffle=True, num_workers=8)

vae = VAE(z_dim=256).cuda()
opt = torch.optim.Adam(vae.parameters(), lr=1e-4)

for epoch in range(100):
    for x,_ in dl:
        x = x.cuda()
        _, recon, kl = vae(x)
        loss = recon + 1.0*kl          # β=1
        opt.zero_grad();  loss.backward();  opt.step()
    print(f"epoch {epoch}: total {loss.item()/len(x):.3f}")
```

────────────────────────────────────────
Inference / Sampling
```python
with torch.no_grad():
    z = torch.randn(1,256).cuda()
    x_gen = vae.dec(z)          # 64×64 image tensor in 0…1
```

────────────────────────────────────────
5.  Practical Tips & Variants  
────────────────────────────────────────
• *Latent scaling* – Your code rescales decoded frames with `((tensor+1)/2)*255`; keep track of ranges (-1…1 vs 0…255).  
• *KL-annealing* or *β-VAE* – linearly ramp β from 0→1 to avoid posterior collapse.  
• *Hierarchical VAEs* – multiple latent groups improve fidelity (used in Stable Diffusion’s SD-VAE).  
• *3-D VAE training* – replace 2-D convs by 3-D convs or factorised (time × space) down-up sampling, train on clips not single frames.  
• *Dual-stage VAE* – Encode each frame (2-D) and a separate VAE for temporal dynamics (3-D); your `DualVAEComponent` ≈ stage-1/2 of such a design.  
• *Freezing the VAE* – In diffusion pipelines you pre-train the VAE once, then **freeze** it while training the diffusion model on latents (`encode=True` to obtain latents, `encode=False` to decode at inference).  
  See the two calls in ClinetoPipeline above.

────────────────────────────────────────
6.  How the Clineto Workflow Hooks a VAE into Diffusion  
────────────────────────────────────────
1. *TEXT / STRUCTURE → Diffusion Transformer* produces **latent videos**.  
2. `DualVAEComponent(encode=False)` decodes those latents to RGB frames (`decoded_frames`).  
3. (Optionally) `Vae3dDecoderComponent` further refines temporal coherence for longer clips.  
4. Post-processing → SSR super-resolution, color correction, video encoding.

By training the VAE separately, the diffusion model can run in compressed latent space (≈ 64× reduction), enabling 1080p 25-frame video on commodity GPUs.

────────────────────────────────────────
7.  Further Reading & Resources  
────────────────────────────────────────
•  Kingma & Welling – “Auto-Encoding Variational Bayes” (2013)  
•  β-VAE (ICLR 2017) – disentanglement via KL weight  
•  VQ-VAE / VQ-VAE-2 – discrete latents (basis of DALL-E 1)  
•  Video VAE – “Stochastic Video Generation with a Learned Prior” (ICML 2018)  
•  “Imagen Video”, “Make-A-Video” – 3-D VAEs as latent compressors  
•  Code: `colligo/models/firefly/imcoding/vae3d`, `dual_vae` folders in your repo

Work through the toy training script first, then trace how `DualVAEComponent` and `Vae3dDecoderComponent` are wired in ClinetoPipeline. That will give you both the *theory* and the *practical wiring* you need to design new T2V/T2I systems.


# Lora

#### Question
I work as a machine learning engineer in Adobe firefly team. I deploy a lot of image and video generation applications. We use diffusion, Dit, transformers, attention, etc The problem is that I don’t know the detail of our networks. it’s been more than 15 years that I have graduated from school and I never learned about these topics in school. I was to learn these stuff. I want to start reading the latest papers and contribute to designing and training these models. In this context I will ask you a few questions. 

Also, the codebase you have indexed has this application, it's in ClinetoPipeline. It's called structure_match or transform_video, you can use it this context to answer my questions as well.

First question, I keep hearing the word "LORA" or Lora weights a lot. What does this mean? does it come from a paper? Can you teach this to me? 

Another context in which I hear the word Lora, is for our structure match application. This is an adaptor on top of our normal text to video application. I know that it's an adaptor on top of the text to video weigths. Do you know how this works? Does this have anything to do with Lora? 

#### Answer
LoRA – short for “Low-Rank Adaptation” – is a light-weight fine-tuning technique that lets you adapt a large, frozen network (language model, diffusion model, vision transformer, …) by training only a few thousand extra parameters instead of all original weights.  
Below is a quick primer, followed by how LoRA frequently shows up in diffusion / T2V systems like the Clineto pipeline and how it differs from the “structure-match” adaptor you mentioned.

────────────────────────────────────────
1.  Where LoRA comes from  
• Paper: “LoRA: Low-Rank Adaptation of Large Language Models”, Xiang Lisa Li & Percy Liang, ICLR 2022.  
• Motivation: Full fine-tuning of a model with N≈10⁸-10¹² weights is costly in memory, storage and time. Parameter-efficient fine-tuning (PEFT) methods try to keep most weights frozen and only learn a small ∆W. Earlier PEFT methods (Adapter-layers, Prefix-tuning, BitFit, etc.) add thousands of extra parameters per layer; LoRA lowers that further and can be merged back into the base weights after training.

────────────────────────────────────────
2.  Basic idea & math  
Consider any weight matrix W₀ ∈ ℝ^{o×i} in a linear or 1×1-conv layer.

Instead of updating W₀ directly, learn a *low-rank* decomposition of the update:

  W’ = W₀ + ΔW with ΔW = B A, rank r ≪ min(o,i)

where  
 • A ∈ ℝ^{r×i}, B ∈ ℝ^{o×r} (r is typically 4–64)  
 • Only A and B are trained; W₀ stays frozen.  
 • A scaling factor α/r is often applied so that the LoRA update has similar magnitude to W₀ during early training.

Parameter count added per layer: r(i + o) instead of o × i.

Because ΔW is added in *series* to the existing projection, LoRA leaves the original forward path intact. At inference you can:

 • Keep A and B as side-car weights (tiny memory hit), or  
 • Fuse them into W₀ once (no run-time cost).

────────────────────────────────────────
3.  Why it is popular  
✓ 1–2 % of original parameters need gradients ⇒ GPU memory & I/O friendly.  
✓ No extra latency after fusion.  
✓ Multiple LoRA “patches” (styles, domains, tasks) can be hot-swapped on the same backbone.  
✓ Works for any linear/conv projection: attention Q/K/V, MLPs, 1×1 conv in U-Nets, etc.

────────────────────────────────────────
4.  LoRA inside diffusion / T2V pipelines  
In image- or video-generation models (Stable Diffusion, DiT, Imagen Video, etc.) LoRA is usually inserted into

• Self-attention and Cross-attention projections (Q/K/V or Q/K, leaving V frozen).  
• Sometimes the feed-forward/MLP layers in transformer blocks.  
• Occasionally 1×1 convs in UNet down/up blocks.

Training looks exactly like normal fine-tuning: run the diffusion loss, but only A & B have `requires_grad=True`. After training you get a LoRA checkpoint that is only a few MB, which the inference code loads and “injects” into the frozen base model.

────────────────────────────────────────
5.  Relation to your “structure-match / transform_video” adaptor  

• In ClinetoPipeline the extra *structure_match* or *transform_video* branch computes **structural_tokens** (see `get_structure_tokens` → `StructureEncoderExecuter`) and feeds them into the Diffusion Transformer as additional conditioning, but the code you attached does **not** explicitly load LoRA matrices. That adaptor is about *conditioning*, not *parameter* changes.

• That said, many teams train the *conditioning-aware* weights themselves with LoRA: they freeze the base T2V model and only LoRA-tune the attention layers so the model learns to respect structure tokens without forgetting text alignment. In such a setup:

  – Base T2V weights = same as production model.  
  – Structure-match LoRA weights = tiny side-car file (often called “adaptor”) that you load right after the backbone.

So whether your structure-match stage “uses LoRA” depends on how the weights were produced and loaded. If your deployment steps mention “lora-rank-…”, “.safetensors” with only `lora_*` keys, or you see code that injects low-rank matrices into attention modules at start-up, then yes, the adaptor itself is implemented as LoRA.

────────────────────────────────────────
6.  Key hyper-parameters you will see  
• rank r (typ. 4, 8, 16, 32)  
• α (the scaling factor; α=r behaves like identity scaling)  
• target_modules (list of layers to LoRA-adapt, e.g. `q_proj,k_proj` or `enc_blocks.*.attn`)  
• dropout p (sometimes inserted between A and B during training)  
• merge_weights bool (fuse at inference or keep separate)

────────────────────────────────────────
7.  Learning resources  
• Original paper (ICLR 2022): tinyurl.com/lora-paper  
• PEFT library (HuggingFace) and `diffusers.load_lora_weights` for diffusion models.  
• “Parameter-Efficient Fine-Tuning Survey”, He et al. 2023 – compares LoRA, Adapters, Prefix-Tuning, etc.

────────────────────────────────────────
8.  TL;DR  
LoRA is a *parameter-efficient* way to fine-tune a frozen backbone by learning low-rank updates to weight matrices. It is ubiquitous in diffusion and T2V because it:

 • requires little GPU memory,  
 • produces tiny checkpoints that can be swapped in/out, and  
 • adds zero inference cost after merging.

Your “structure-match” adaptor can be implemented with or without LoRA; if its weights are tiny and loaded as low-rank matrices into the attention layers, then it is indeed a LoRA patch sitting on top of the base text-to-video model.
